# 공공데이터 수집에서 막히는 지점 실습 노트북
DataPopcorn 테크블로그 글 「공공데이터 수집에서 막히는 6가지와 해결법」의 실습 코드입니다.

1. 공공데이터포털 인증키: 다시 인코딩하면 `SERVICE_KEY_IS_NOT_REGISTERED_ERROR`
2. 공정위 브랜드별 가맹점 API 전체 받기 (페이지 넘기기, 중복 행 처리)
3. PDF 통계표: `-layout` 텍스트의 빈칸 함정과 `-bbox-layout` 좌표 파싱
4. 교차검증: 회차 합 vs 연간 합계, 다른 파일에 실린 같은 해 값 비교

필요한 것: 공공데이터포털 회원가입 후 **공정거래위원회_가맹정보_브랜드별 가맹점 현황 제공 서비스** 활용신청(개발계정은 즉시 승인).

## 1. 인증키는 '그대로' 붙입니다
마이페이지의 **일반 인증키(Encoding)** 값을 입력합니다. 키는 화면에 출력하지 않습니다.

In [ ]:
import os, json, getpass, urllib.request, urllib.parse, urllib.error

# 공공데이터포털 마이페이지 > 활용신청 상세의 '일반 인증키(Encoding)' 값을 그대로 붙여 넣습니다.
SERVICE_KEY = os.environ.get('DATA_GO_KR_KEY') or getpass.getpass('일반 인증키(Encoding): ')
SERVICE_KEY = SERVICE_KEY.strip()
BASE = 'https://apis.data.go.kr/1130000/FftcBrandFrcsStatsService/getBrandFrcsStats'

def get(url):
    try:
        with urllib.request.urlopen(url, timeout=60) as r:
            return r.status, r.read().decode('utf-8', 'replace')
    except urllib.error.HTTPError as e:
        return e.code, e.read().decode('utf-8', 'replace')

rest = {'pageNo': 1, 'numOfRows': 5, 'resultType': 'json', 'yr': 2025}

# (X) 이미 인코딩된 키를 한 번 더 인코딩: %가 %25로 바뀌어 키가 달라집니다.
wrong = BASE + '?' + urllib.parse.urlencode({'serviceKey': SERVICE_KEY, **rest})
code, body = get(wrong)
print('잘못된 방법 ->', code, body[:120].replace(SERVICE_KEY, '***'))

# (O) Encoding 키는 손대지 않고 문자열로 그대로 붙입니다.
right = f'{BASE}?serviceKey={SERVICE_KEY}&' + urllib.parse.urlencode(rest)
code, body = get(right)
print('올바른 방법 ->', code, json.loads(body)['resultMsg'], '전체', json.loads(body)['totalCount'], '건')

## 2. 공정위 가맹점 API 전체 받기
`yr=2025`는 **2024년 말** 기준 값입니다(직전 사업연도 말). 1,000건씩 페이지를 넘기고, 같은 해 같은 브랜드가 여러 행이면 최댓값을 씁니다.

In [ ]:
import time
rows, page = [], 1
while True:
    q = urllib.parse.urlencode({'pageNo': page, 'numOfRows': 1000, 'resultType': 'json', 'yr': 2025})
    code, body = get(f'{BASE}?serviceKey={SERVICE_KEY}&{q}')
    items = json.loads(body).get('items') or []
    rows += items
    if not items or len(rows) >= int(json.loads(body)['totalCount']):
        break
    page += 1
    time.sleep(0.5)   # 요청 사이 간격
print('받은 행', len(rows))
best = {}
for x in rows:
    if '치킨' in x['indutyMlsfcNm'] and int(x['frcsCnt']) > 0:
        best[x['brandNm']] = max(best.get(x['brandNm'], 0), int(x['frcsCnt']))   # 같은 해 중복 행은 최댓값
for b, n in sorted(best.items(), key=lambda kv: -kv[1])[:5]:
    print(f'{b:20s} {n:,}곳')

## 3. PDF 통계표 준비
`topik.go.kr` 파일은 스크립트로 바로 받으면 PDF 대신 1.6KB짜리 안내 HTML이 내려옵니다. 브라우저에서 받아 올립니다.

1. 브라우저로 https://www.topik.go.kr 메인을 연 뒤, 같은 브라우저 주소창에 아래 두 주소를 엽니다.
   - https://www.topik.go.kr/topik/cm/comm/downloadContent.do?filePath=pdf&fileName=2021%7E2025+TOPIK+Download.pdf
   - https://www.topik.go.kr/topik/cm/comm/downloadContent.do?filePath=pdf&fileName=2017%7E2021+TOPIK+Download.pdf
2. 아래 셀을 실행하고 받은 두 파일을 업로드합니다(파일 이름의 연도로 자동 구분).

In [ ]:
import os, subprocess
subprocess.run('apt-get update -qq && apt-get install -y -qq poppler-utils', shell=True, capture_output=True)
print('pdftotext 설치:', subprocess.run(['which', 'pdftotext'], capture_output=True, text=True).stdout.strip())
need = ['topik_2021_2025.pdf', 'topik_2017_2021.pdf']
if not all(os.path.exists(f) for f in need):
    from google.colab import files
    for name in files.upload():
        target = 'topik_2017_2021.pdf' if '2017' in name else 'topik_2021_2025.pdf'
        os.replace(name, target)
for f in need:
    print(f, os.path.getsize(f), 'bytes', open(f, 'rb').read(5))

## 4. 함정: `-layout` 텍스트를 공백으로 자르기
2024년부터 표의 빈칸이 `-` 대신 공백이라, 공백 기준으로 자르면 값이 앞 회차로 당겨집니다.

In [ ]:
# 함정 확인: -layout 텍스트를 공백으로 자르면 빈칸이 사라져 열이 밀립니다.
subprocess.run(['pdftotext', '-layout', 'topik_2021_2025.pdf', 'layout.txt'], check=True)
txt = open('layout.txt', encoding='utf-8').read()
sec = txt.split('2024년도 시행 현황')[1]
line = next(l for l in sec.splitlines() if l.strip().startswith('중국'))
tokens = line.split()[1:]
print('2024 중국 행 숫자 토큰 수:', len(tokens), '(회차 6개 x 6열 + 합계 6열 + 총계 = 43칸이어야 함)')
print('공백 기준 첫 토큰:', tokens[:3], '<- 실제로는 93회 값인데 첫 회차(92회) 자리로 당겨짐')

## 5. 해결: `-bbox-layout` 좌표로 열 맞추기
단어마다 x·y 좌표가 나오므로, 머리글 '지원자/응시자/합격자'의 x 위치를 열로 삼고 숫자를 가장 가까운 열에 배정합니다.

In [ ]:
import re, html, subprocess

def bbox_rows(pdf):
    """pdftotext -bbox-layout 결과를 페이지별 '같은 높이의 단어 묶음(행)'으로 바꿉니다."""
    out = pdf.replace('.pdf', '.bbox.html')
    subprocess.run(['pdftotext', '-bbox-layout', pdf, out], check=True)
    s = open(out, encoding='utf-8').read()
    for pg in re.findall(r'<page[^>]*>(.*?)</page>', s, re.S):
        rows = {}
        for a, b, c, d, t in re.findall(r'<word xMin="([\d.]+)" yMin="([\d.]+)" xMax="([\d.]+)" yMax="([\d.]+)">(.*?)</word>', pg):
            w = (float(a), float(b), float(c), float(d), html.unescape(t))
            rows.setdefault(round((w[1] + w[3]) / 4), []).append(w)   # y 중심 2pt 단위로 묶기
        yield [sorted(v) for _, v in sorted(rows.items())]

def parse_topik(pdf):
    """국가별·회차별 지원자(TOPIK I + II)와, 표 끝의 연간 합계 열을 함께 돌려줍니다."""
    R, Y = {}, {}
    year = rounds = cols = None
    for rows in bbox_rows(pdf):
        m = re.search(r'(20\d\d)년도', ' '.join(w[4] for r in rows for w in r))
        if m:
            year, rounds, cols = int(m.group(1)), None, None
        for r in rows:
            toks = [w[4] for w in r]
            if any(re.fullmatch(r'\d+회', t) for t in toks):
                rounds = [int(t[:-1]) for t in toks if re.fullmatch(r'\d+회', t)]
            if toks.count('지원자') >= 4:   # 머리글 행의 '지원자/응시자/합격자' x 좌표 = 열 위치
                cols = [(w[0] + w[2]) / 2 for w in r if w[4] in ('지원자', '응시자', '합격자')]
        if not (rounds and cols):
            continue
        for r in rows:
            name = ' '.join(w[4] for w in r if not re.fullmatch(r'[\d,]+|-', w[4]))
            nums = [w for w in r if re.fullmatch(r'[\d,]+', w[4])]
            if not nums or not name or re.search(r'국가|지원자|기준|시행|현황|회|TOPIK|단위', name):
                continue
            vals = {}
            for w in nums:   # 숫자의 x 중심과 가장 가까운 머리글 열에 배정 (빈칸은 자연히 비어 있음)
                xc = (w[0] + w[2]) / 2
                j = min(range(len(cols)), key=lambda k: abs(cols[k] - xc))
                if abs(cols[j] - xc) < 25:
                    vals[j] = int(w[4].replace(',', ''))
            for k, rd in enumerate(rounds):   # 회차마다 6열: I(지원·응시·합격) II(지원·응시·합격)
                v = vals.get(k * 6, 0) + vals.get(k * 6 + 3, 0)
                if v:
                    R.setdefault(year, {}).setdefault(name, {})[rd] = v
            n = len(rounds)
            Y.setdefault(year, {})[name] = vals.get(n * 6, 0) + vals.get(n * 6 + 3, 0)
    return R, Y

R_new, Y_new = parse_topik('topik_2021_2025.pdf')
print('연도', sorted(R_new))

## 6. 교차검증
파싱 결과를 믿기 전에 두 가지를 맞춰 봅니다. 하나라도 어긋나면 파서를 고칩니다.

In [ ]:
# 검증 1: 회차별 값을 더한 합계 == PDF에 인쇄된 연간 합계 열
bad = [(y, c, sum(R_new[y][c].values()), Y_new[y].get(c))
       for y in R_new for c in R_new[y] if sum(R_new[y][c].values()) != Y_new[y].get(c)]
print('회차 합 vs 연간 합계 불일치:', len(bad), bad[:5])

# 검증 2: 다른 파일(2017~2021)에 실린 같은 해(2021) 값과 비교
R_old, _ = parse_topik('topik_2017_2021.pdf')
for c in ['중국', '일본', '베트남', '몽골']:
    a, b = sum(R_old[2021][c].values()), sum(R_new[2021][c].values())
    print(f'2021 {c}: 옛 파일 {a:,} / 새 파일 {b:,}', '일치' if a == b else '불일치')

# 결과: 해외 시행국 지원자 TOP 3 (한국 국내 시행·총합계 제외)
for y in sorted(R_new):
    top = sorted(((c, sum(v.values())) for c, v in R_new[y].items() if c not in ('한국', '총합계')), key=lambda kv: -kv[1])[:3]
    print(y, ' / '.join(f'{c} {n:,}' for c, n in top))